# ML Model --- Airplane Crash Data --- Thomas Hughes 

---

## * Initial setup *

In [132]:
# import libraries
import pandas as pd
import numpy as np

In [133]:
!pip install pycountry
import pycountry

In [89]:
# import csv
df = pd.read_csv("../Data/Airplane_Crashes_and_Fatalities_Since_1908_data.csv")

## 1. Data Quality: Pre-processing

In [69]:
# first evaluation
df.head(5)

,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary
0,07/12/1912,06:30,"AtlantiCity, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...
1,08/06/1913,NaN,"Victoria, British Columbia, Canada",Private,-,NaN,Curtiss seaplane,NaN,NaN,1.0,1.0,0.0,The first fatal airplane accident in Canada oc...
2,09/09/1913,18:30,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,14.0,0.0,The airship flew into a thunderstorm and encou...
3,03/05/1915,01:00,"Tienen, Belgium",Military - German Navy,NaN,NaN,Zeppelin L-8 (airship),NaN,NaN,41.0,21.0,0.0,Crashed into trees while attempting to land af...
4,09/03/1915,15:20,"Off Cuxhaven, Germany",Military - German Navy,NaN,NaN,Zeppelin L-10 (airship),NaN,NaN,19.0,19.0,0.0,"Exploded and burned near Neuwerk Island, when..."


In [70]:
df.shape

(4741, 13)

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4741 entries, 0 to 4740
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          4741 non-null   object 
 1   Time          2741 non-null   object 
 2   Location      4724 non-null   object 
 3   Operator      4726 non-null   object 
 4   Flight #      967 non-null    object 
 5   Route         3196 non-null   object 
 6   Type          4717 non-null   object 
 7   Registration  4441 non-null   object 
 8   cn/In         3643 non-null   object 
 9   Aboard        4722 non-null   float64
 10  Fatalities    4730 non-null   float64
 11  Ground        4721 non-null   float64
 12  Summary       4388 non-null   object 
dtypes: float64(3), object(10)
memory usage: 481.6+ KB


In [72]:
df.isna().sum().to_frame(name="Number of nulls")

,Number of nulls
Date,0
Time,2000
Location,17
Operator,15
Flight #,3774
Route,1545
Type,24
Registration,300
cn/In,1098
Aboard,19


#### 'Date' assessment

In [73]:
# date assessment
# from .head() we can see that format is MM/DD/YYYY

print("Datatype BEFORE transformation:", df["Date"].dtype)
print("Number of null dates BEFORE transformation:", df["Date"].isna().sum())

# transform object -> datetime64
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

print("----------------------------------------------")
print("Datatype AFTER transformation:", df["Date"].dtype)
print("Number of null dates AFTER object -> datetime64:", df["Date"].isna().sum())

Datatype BEFORE transformation: object
Number of null dates BEFORE transformation: 0
----------------------------------------------
Datatype AFTER transformation: datetime64[ns]
Number of null dates AFTER object -> datetime64: 0


In [74]:
print("Earliest date:", df["Date"].min())
print("Latest date:", df["Date"].max())
print("--------------------------------")
if df["Date"].max() < pd.Timestamp.today():
    print("All dates are earlier than today")

Earliest date: 1912-07-12 00:00:00
Latest date: 2009-06-07 00:00:00
--------------------------------
All dates are earlier than today


In [75]:
# are dates in descending order?
df["Date"].is_monotonic_increasing

False

#### 'Time' assessment

In [90]:
# time assesssment
null_percent = ((df["Time"].isna().sum()/df.shape[0])*100).round(1)
print("Percentage of nulls:", null_percent, "%")

Percentage of nulls: 42.2 %


In [91]:
# first check, find values which do not have a string length of 5 (therefore must be invalid)
df["Time"].dropna().str.len().value_counts().to_frame(name="Count")

,Count
Time,
5,2726
4,7
7,5
6,3


In [92]:
invalid_length_times = df["Time"].dropna()[
    df["Time"].dropna().str.len() != 5
]

display(invalid_length_times)

168     c: 1:00
188     c:17:00
202     c: 2:00
636        1:30
1307     c16:50
2323    c:09:00
2924     114:20
3035     c14:30
3170       0943
3216       1:00
3859       2:40
4340    c: 9:40
4350       2:00
4351       8:02
4638       9:30
Name: Time, dtype: object

In [105]:
# more thorough check to find times which are string length 5 but an incorrect format
valid_time_mask = df["Time"].dropna().str.match(r"^\d{2}:\d{2}$")

print("Valid time format:", valid_time_mask.sum())
print("Invalid time format:", (~valid_time_mask).sum())

invalid_time = df["Time"].dropna()[
    ~df["Time"].dropna().str.match(r"^\d{2}:\d{2}$")
]

print("Invalid time value:", invalid_time)

Valid time format: 2725
Invalid time format: 1
Invalid time value: 4332    22'08
Name: Time, dtype: object


In [106]:
# replacing all invalid values manually using a dictionary:

time_corrections = {
    "c: 1:00": "01:00",
    "c:17:00": "17:00",
    "c: 2:00": "02:00",
    "1:30": "01:30",
    "c16:50": "16:50",
    "c:09:00": "09:00",
    "114:20": np.nan,   # unclear invalid value - either number could have been mistakenly added
    "c14:30": "14:30",
    "0943": "09:43",
    "1:00": "01:00",
    "2:40": "02:40",
    "c: 9:40": "09:40",
    "2:00": "02:00",
    "8:02": "08:02",
    "9:30": "09:30",
    "22'08": "22:08"
}

In [107]:
# apply corrections
df["Time"] = df["Time"].replace(time_corrections)

In [111]:
# convert datatype from 'object' to 'datetime'
print("Data type BEFORE transformation:", df["Time"].dtype)
df["Time"] = pd.to_datetime(
    df["Time"],
    format="%H:%M",
    errors="coerce"
)
print("Data type AFTER transformation:", df["Time"].dtype)

Data type BEFORE transformation: object
Data type AFTER transformation: datetime64[ns]


In [112]:
# create new categorical column

# create placeholder
df["Time_Period"] = "Unknown"

# create new column Time_Period which buckets:
# 00:00 -> 05:59 = Night, 06:00 -> 11:59 = Morning, 12:00 -> 17:59 = Afternoon, 18:00 -> 23:59 = Evening, Missing values = Unknown
df.loc[df["Time"].dt.hour.between(0, 5), "Time_Period"] = "Night"
df.loc[df["Time"].dt.hour.between(6, 11), "Time_Period"] = "Morning"
df.loc[df["Time"].dt.hour.between(12, 17), "Time_Period"] = "Afternoon"
df.loc[df["Time"].dt.hour.between(18, 23), "Time_Period"] = "Evening"

In [113]:
# check to make sure that categories have been assigned, and there are no nulls
display(df["Time_Period"].value_counts(dropna=False))

Time_Period
Unknown      2015
Afternoon     883
Morning       782
Evening       722
Night         339
Name: count, dtype: int64

#### 'Location' assessment 

In [114]:
null_percent = ((df["Location"].isna().sum()/df.shape[0])*100).round(1)
print("Percentage of nulls:", null_percent, "%")

Percentage of nulls: 0.4 %


In [119]:
df.head(4)

,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary,Time_Period
0,07/12/1912,1900-01-01 06:30:00,"AtlantiCity, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...,Morning
1,08/06/1913,NaT,"Victoria, British Columbia, Canada",Private,-,NaN,Curtiss seaplane,NaN,NaN,1.0,1.0,0.0,The first fatal airplane accident in Canada oc...,Unknown
2,09/09/1913,1900-01-01 18:30:00,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,14.0,0.0,The airship flew into a thunderstorm and encou...,Evening
3,03/05/1915,1900-01-01 01:00:00,"Tienen, Belgium",Military - German Navy,NaN,NaN,Zeppelin L-8 (airship),NaN,NaN,41.0,21.0,0.0,Crashed into trees while attempting to land af...,Night


In [120]:
# isolating everything after the final comma
df["Country"] = df["Location"].str.split(",").str[-1].str.strip()

In [147]:
display(df[["Location", "Country"]].head(10))

,Location,Country
0,"AtlantiCity, New Jersey",Jersey
1,"Victoria, British Columbia, Canada",Canada
2,Over the North Sea,Over the North Sea
3,"Tienen, Belgium",Belgium
4,"Off Cuxhaven, Germany",Germany
5,"Near Jambol, Bulgeria",Bulgeria
6,"Billericay, England",England
7,"Mainz, Germany",Germany
8,"Off West Hartlepool, England",England
9,"Near Gent, Belgium",Belgium


In [289]:
display(
    df.loc[
        df["Country"].str.contains("Virgin ", case=False, na=False),
        ["Location", "Country"]
    ]
)

,Location,Country
2026,"St. Thomas, Virgin Islands",Virgin Islands
2136,"Near St. Croix, US Virgin Islands",Virgin Islands
2183,"St. Thomas, Virgin Islands",Virgin Islands
2470,"Off Saint Thomas, U.S. Virgin Islands",Virgin Islands
2577,"St. Croix, Virgin Islands",Virgin Islands
2592,"St. Thomas, Virgin Islands",Virgin Islands
2729,"St. Thomas, Virgin Islands",Virgin Islands
2817,"St. Croix, Virgin Islands",Virgin Islands
2964,"St. Thomas, U.S. Virgin Islands",Virgin Islands
3684,"St. Thomas, Virgin Islands",Virgin Islands


In [122]:
df.groupby("Country").size()

Country
110 miles West of Ireland               1
325 miles east of Wake Island           1
AK                                      1
Afghanistan                            30
Afghanstan                              2
                                       ..
Zimbabwe                                2
off Angola                              1
off Australia                           1
off Bermuda                             1
off the Philippine island of Elalat     1
Length: 474, dtype: int64

In [341]:
# set of pycountry countries
countries = {country.name for country in pycountry.countries}
#countries

In [321]:
# set of country names from pycountry
#countries = {country.name for country in pycountry.countries}
#countries

# create a lookup table (dictionary) of pycountry names
lookup = {
    country.name: country.name
    for country in pycountry.countries
}

# amend the dictionary with alias'
lookup.update({

    # adding in common/historic names which do not follow ISO
    
    "Viet Nam": "Viet Nam",
    "Vietnam": "Viet Nam",
    "Russia": "Russian Federation",
    "USSR": "Russian Federation",
    "UAE": "United Arab Emirates",
    "Syria": "Syrian Arab Republic",
    "Turkey": "Türkiye",
    "Iran": "Iran, Islamic Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Bolivia": "Bolivia, Plurinational State of",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Yugoslavia": "Yugoslavia",
    "Soviet Union": "Russian Federation",
    "Zaire": "Congo",
    "Zaïre": "Congo",
    "Burma": "Myanmar",
    "Macedonia": "North Macedonia",
    "Rhodesia": "Zimbabwe",
    "Tanganyika": "Tanzania, United Republic of",
    "French Equatorial Africa": "French Equatorial Africa",
    "French West Africa": "French West Africa",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Bosnia": "Bosnia and Herzegovina",
    "Timor": "Timor-Leste",
    "East Timor": "Timor-Leste",
    "Tanzania": "Tanzania, United Republic of",
    "Trinidad": "Trinidad and Tobago",
    "Djibouti": "Djibouti",
    "Mauritania": "Mauritania",
    "Khmer Republic": "Cambodia",
    "Upper Volta": "Burkina Faso",
    "Surinam": "Suriname",
    "Kirghizia": "Kyrgyzstan",
    "Czech Republic": "Czechia",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Saskatchewan": "Canada",
    "Manitoba": "Canada",
    "British Columbia": "Canada",
    "Tasmania": "Tasmania",
    "Caribbean": "Caribbean",
    "Cape Verde Islands": "Cabo Verde",
    "Sainte Lucia Island": "Saint Lucia",
    "Grenadines Islands": "Grenadines",
    "Antigua": "Antigua and Barbuda",
    "Brunei": "Brunei Darussalam",
    "Chechnya": "Russian Federation",
    "Ivory Coast": "Côte d'Ivoire",
    "Malagasy Republic": "Madagascar",
    "UAR": "Egypt",
    "Malaya": "Malaysia",
    "French Equitorial Africa": "French Equatorial Africa",
    "Reunion": "Réunion",
    "Moldova": "Moldova, Republic of",

    # UK states
    
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Northern Ireland": "United Kingdom",

    # Territories/ Islands:

    "Newfoundland": "Newfoundland",
    "Virgin Islands": "Virgin Islands, U.S.",
    "British Virgin Islands": "Virgin Islands, British",
    "U.S. Virgin Islands": "Virgin Islands, U.S.",
    "US Virgin Islands": "Virgin Islands, U.S.",
    "Canary Islands": "Canary Islands",
    "Azores": "Azores",
    "West Indies": "West Indies",
    "Comoro Islands": "Comoros",
    "Turks & Caicos Islands": "Turks and Caicos Islands",
    "Leeward Islands": "Leeward Islands",
    "Crete": "Crete",
    "Okinawa": "Okinawa",
    "Eugene Island": "Eugene Island",
    "Islay Island": "Islay Island",
    "Mariana Islands": "Mariana Islands",
    "Isle of man": "Isle of Man",
    "Sao Tomé & Principe": "Sao Tome and Principe",
    "Grenadines": "Grenadines",
    "Borneo": "Borneo",
    "Great Inagua": "Great Inagua",
    "Bimini": "Bimini",
    "Off  Bimini": "Bimini",

    # USA states

    "Alabama": "Alabama",
    "Alaska": "Alaska",
    "Arizona": "Arizona",
    "Arkansas": "Arkansas",
    "California": "California",
    "Colorado": "Colorado",
    "Connecticut": "Connecticut",
    "Delaware": "Delaware",
    "Florida": "Florida",
    "Georgia": "Georgia",
    "Hawaii": "Hawaii",
    "Idaho": "Idaho",
    "Illinois": "Illinois",
    "Indiana": "Indiana",
    "Iowa": "Iowa",
    "Kansas": "Kansas",
    "Kentucky": "Kentucky",
    "Louisiana": "Louisiana",
    "Maine": "Maine",
    "Maryland": "Maryland",
    "Massachusetts": "Massachusetts",
    "Michigan": "Michigan",
    "Minnesota": "Minnesota",
    "Mississippi": "Mississippi",
    "Missouri": "Missouri",
    "Montana": "Montana",
    "Nebraska": "Nebraska",
    "Nevada": "Nevada",
    "New Hampshire": "New Hampshire",
    "New Jersey": "New Jersey",
    "New Mexico": "New Mexico",
    "New York": "New York",
    "North Carolina": "North Carolina",
    "North Dakota": "North Dakota",
    "Ohio": "Ohio",
    "Oklahoma": "Oklahoma",
    "Oregon": "Oregon",
    "Pennsylvania": "Pennsylvania",
    "Rhode Island": "Rhode Island",
    "South Carolina": "South Carolina",
    "South Dakota": "South Dakota",
    "Tennessee": "Tennessee",
    "Texas": "Texas",
    "Utah": "Utah",
    "Vermont": "Vermont",
    "Virginia": "Virginia",
    "Washington": "Washington",
    "West Virginia": "West Virginia",
    "Wisconsin": "Wisconsin",
    "Wyoming": "Wyoming",

    # Water:

    # Oceans
    "Atlantic Ocean": "Atlantic Ocean",
    "AtlanticOcean": "Atlantic Ocean",
    "AtlantiOcean": "Atlantic Ocean",
    "North AtlanticOcean": "Atlantic Ocean",
    "South AtlanticOcean": "Atlantic Ocean",

    "Pacific Ocean": "Pacific Ocean",
    "PacificOcean": "Pacific Ocean",
    "PacifiOcean": "Pacific Ocean",
    "North PacificOcean": "Pacific Ocean",

    # Seas
    "North Sea": "North Sea",
    "Baltic Sea": "Baltic Sea",
    "Black Sea": "Black Sea",
    "Mediterranean Sea": "Mediterranean Sea",
    "Philippine Sea": "Philippine Sea",
    "Andaman Sea": "Andaman Sea",

    # Gulfs
    "Persian Gulf": "Persian Gulf",
    "Gulf of Sirte": "Gulf of Sirte",
    "Gulf of Tonkin": "Gulf of Tonkin",

    # Channels / Straits
    "English Channel": "English Channel",
    "Formosa Strait": "Taiwan Strait",

    # "Over ..." variants
    "Over the AtlanticOcean": "Atlantic Ocean",
    "Over the AtlantiOcean": "Atlantic Ocean",
    "Over the North Atlantic": "Atlantic Ocean",
    "North Atlantic": "Atlantic Ocean",   
    "Over the PacificOcean": "Pacific Ocean",
    "Over the PacifiOcean": "Pacific Ocean",
    "Over the North PacificOcean": "Pacific Ocean",
    "Over the Mediterranean Sea": "Mediterranean Sea",
    "Over the North Sea": "North Sea",
    "Over the English Channel": "English Channel",
    "Over the Andaman Sea": "Andaman Sea",

    # spelling mistakes and ambiguous fixes

    "Massachusett": "Massachusetts",
    "Massachutes": "Massachusetts",
    "Tennesee": "Tennessee",
    "Wisconson": "Wisconsin",
    "Alaksa": "Alaska",
    "Alakska": "Alaska",
    "Washingon": "Washington",
    "Afghanstan": "Afghanistan",
    "Minnisota": "Minnesota",
    "Boliva": "Bolivia",
    "Mauretania": "Mauritania",
    "Coloado": "Colorado",
    "Napal": "Nepal",
    "Philipines": "Philippines",
    "Phillipines": "Philippines",
    "Hunary": "Hungary",
    "Calilfornia": "California",
    "Cailifornia": "California",
    "Djbouti": "Djibouti",
    "Morroco": "Morocco",
    "Morrocco": "Morocco",
    "Yugosalvia": "Yugoslavia",
    "Manmar": "Myanmar",
    "Hati": "Haiti",
    "Domincan Republic": "Dominican Republic",
    "DemocratiRepubliCogo": "Congo, The Democratic Republic of the",
    "Airzona": "Arizona",
    "Oklohoma": "Oklahoma",
    "Thiland": "Thailand",
    "Sierre Leone": "Sierra Leone",
    "Bulgeria": "Bulgaria",
    "Bugaria": "Bulgaria",
    "Ilinois": "Illinois",
    "Arazona": "Arizona",
    "Deleware": "Delaware",
    "Romainia": "Romania",
    "Jamacia": "Jamaica",
    "Aregntina": "Argentina",
    "Baangladesh": "Bangladesh",
    "Mocambique": "Mozambique",
    "Louisana": "Louisiana",
    "South Dekota": "South Dakota",
    "Midway Island Naval Air Station": "United States Minor Outlying Islands",

    #abbreviations of states
    "AK": "Alaska",
    "CA": "California",
    "GA": "Georgia",
    "HI": "Hawaii",
    "NY": "New York",
    "WY": "Wyoming",
    "UK": "United Kingdom",
    "D.C.": "Washington D.C.",

    #cities/areas within countries
    "Algiers": "Algeria",
    "Vienna": "Austria",
    "Amsterdam": "Netherlands",
    "London": "United Kingdom",
    "Buenos Aires": "Argentina",
    "Sao Tomé": "Sao Tome and Principe",
    "Milford Sound": "New Zealand",
    "Near Moscow": "Russian Federation",
    "Near Karkov": "Ukraine"
})

In [322]:
# sorting order
search_terms = sorted(
    lookup.keys(),
    key=len,
    reverse=True
)

In [323]:
# extraction function
def extract_country(location):

    if pd.isna(location):
        return location

    location_lower = location.lower()

    for term in search_terms:
        if term.lower() in location_lower:
            return lookup[term]

    return location

df["Country"] = df["Country"].apply(extract_country)

In [325]:
# find left over values
display(
    df.loc[
        ~df["Country"].isin(lookup),
        "Country"
    ].value_counts()
)

Country
Over the Mediterranean    1
Off Irish coast           1
Near Nag                  1
BO                        1
Name: count, dtype: int64

In [327]:
# turing the remaining 4 values to null (Over the Mediterranean, Off Irish coast, Near Nag, BO) 
df.loc[
    df["Country"].isin([
        "Over the Mediterranean",
        "Off Irish coast",
        "Near Nag",
        "BO"
    ]),
    "Country"
] = np.nan

In [339]:
# further reduce to us states, continent, body of water, sslands/territories, unknowns


us_states = {
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado",
    "Connecticut","Delaware","Florida","Georgia","Hawaii","Idaho",
    "Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana",
    "Maine","Maryland","Massachusetts","Michigan","Minnesota",
    "Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York",
    "North Carolina","North Dakota","Ohio","Oklahoma","Oregon",
    "Pennsylvania","Rhode Island","South Carolina","South Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington",
    "West Virginia","Wisconsin","Wyoming","Washington D.C."
}

df.loc[df["Country"].isin(us_states), "Country"] = "United States"

# create region column
region_lookup = {

    # North America
    "United States": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    "Bahamas": "North America",
    "Cuba": "North America",
    "Jamaica": "North America",
    "Dominican Republic": "North America",
    "Haiti": "North America",
    "Trinidad and Tobago": "North America",
    "Antigua and Barbuda": "North America",
    "Saint Lucia": "North America",
    "Côte d'Ivoire": "Africa",
    "Puerto Rico": "North America",
    "Honduras": "North America",
    "Guatemala": "North America",
    "Guadeloupe": "North America",
    "Nicaragua": "North America",
    "Panama": "North America",
    "Virgin Islands, U.S.": "North America",
    "Costa Rica": "North America",
    "Greenland": "North America",
    "Guam": "North America",
    "El Salvador": "North America",
    "Bermuda": "North America",
    "Turks and Caicos Islands": "North America",
    "Barbados": "North America",
    "American Samoa": "North America",
    "Martinique": "North America",
    "Dominica": "North America",
    "Belize": "North America",

    # South America
    "Argentina": "South America",
    "Brazil": "South America",
    "Chile": "South America",
    "Colombia": "South America",
    "Ecuador": "South America",
    "Guyana": "South America",
    "Paraguay": "South America",
    "Peru": "South America",
    "Suriname": "South America",
    "Uruguay": "South America",
    "Venezuela, Bolivarian Republic of": "South America",
    "Bolivia, Plurinational State of": "South America",

    # Europe
    "United Kingdom": "Europe",
    "Ireland": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Austria": "Europe",
    "Belgium": "Europe",
    "Netherlands": "Europe",
    "Switzerland": "Europe",
    "Italy": "Europe",
    "Spain": "Europe",
    "Portugal": "Europe",
    "Norway": "Europe",
    "Sweden": "Europe",
    "Finland": "Europe",
    "Denmark": "Europe",
    "Poland": "Europe",
    "Czechia": "Europe",
    "Romania": "Europe",
    "Hungary": "Europe",
    "Bulgaria": "Europe",
    "Greece": "Europe",
    "Croatia": "Europe",
    "Serbia": "Europe",
    "Bosnia and Herzegovina": "Europe",
    "North Macedonia": "Europe",
    "Ukraine": "Europe",
    "Russian Federation": "Europe",
    "Yugoslavia": "Europe",
    "Jersey": "Europe",
    "Lithuania": "Europe",
    "Moldova, Republic of": "Europe",
    "Slovakia": "Europe",
    "Malta": "Europe",
    "Estonia": "Europe",
    "Iceland": "Europe",
    "Gibraltar": "Europe",
    "Cyprus": "Europe",
    "Luxembourg": "Europe",
    "Latvia": "Europe",
    "Albania": "Europe",
    "Slovenia": "Europe",
    "Isle of Man": "Europe",

    # Asia
    "China": "Asia",
    "Japan": "Asia",
    "India": "Asia",
    "Pakistan": "Asia",
    "Afghanistan": "Asia",
    "Thailand": "Asia",
    "Myanmar": "Asia",
    "Malaysia": "Asia",
    "Indonesia": "Asia",
    "Philippines": "Asia",
    "Singapore": "Asia",
    "Viet Nam": "Asia",
    "Lao People's Democratic Republic": "Asia",
    "Cambodia": "Asia",
    "Korea, Republic of": "Asia",
    "Korea, Democratic People's Republic of": "Asia",
    "Taiwan, Province of China": "Asia",
    "Türkiye": "Asia",
    "Iran, Islamic Republic of": "Asia",
    "Iraq": "Asia",
    "Saudi Arabia": "Asia",
    "United Arab Emirates": "Asia",
    "Jordan": "Asia",
    "Israel": "Asia",
    "Syrian Arab Republic": "Asia",
    "Yemen": "Asia",
    "Kazakhstan": "Asia",
    "Kyrgyzstan": "Asia",
    "Nepal": "Asia",
    "Sri Lanka": "Asia",
    "Timor-Leste": "Asia",
    "Brunei Darussalam": "Asia",
    "Hong Kong": "Asia",
    "Lebanon": "Asia",
    "Mongolia": "Asia",
    "Uzbekistan": "Asia",
    "Azerbaijan": "Asia",
    "Bangladesh": "Asia",
    "Armenia": "Asia",
    "Qatar": "Asia",
    "Oman": "Asia",
    "Kuwait": "Asia",
    "Bhutan": "Asia",
    "Turkmenistan": "Asia",
    "Tajikistan": "Asia",
    "Bahrain": "Asia",

    # Africa
    "Algeria": "Africa",
    "Angola": "Africa",
    "Botswana": "Africa",
    "Burkina Faso": "Africa",
    "Burundi": "Africa",
    "Cameroon": "Africa",
    "Cabo Verde": "Africa",
    "Central African Republic": "Africa",
    "Chad": "Africa",
    "Comoros": "Africa",
    "Congo": "Africa",
    "Congo, The Democratic Republic of the": "Africa",
    "Djibouti": "Africa",
    "Egypt": "Africa",
    "Equatorial Guinea": "Africa",
    "Ethiopia": "Africa",
    "Gabon": "Africa",
    "Ghana": "Africa",
    "Kenya": "Africa",
    "Libya": "Africa",
    "Madagascar": "Africa",
    "Malawi": "Africa",
    "Mali": "Africa",
    "Mauritania": "Africa",
    "Morocco": "Africa",
    "Mozambique": "Africa",
    "Namibia": "Africa",
    "Nigeria": "Africa",
    "Rwanda": "Africa",
    "Senegal": "Africa",
    "Sierra Leone": "Africa",
    "Somalia": "Africa",
    "South Africa": "Africa",
    "Sudan": "Africa",
    "Tanzania, United Republic of": "Africa",
    "Tunisia": "Africa",
    "Uganda": "Africa",
    "Zambia": "Africa",
    "Zimbabwe": "Africa",
    "French Equatorial Africa": "Africa",
    "French West Africa": "Africa",
    "Réunion": "Africa",
    "Guinea": "Africa",
    "Gambia": "Africa",
    "Niger": "Africa",
    "Liberia": "Africa",
    "Sao Tome and Principe": "Africa",
    "Eritrea": "Africa",
    "Lesotho": "Africa",
    "Benin": "Africa",

    # Oceania
    "Australia": "Oceania",
    "New Zealand": "Oceania",
    "Papua New Guinea": "Oceania",
    "Fiji": "Oceania",
    "Samoa": "Oceania",
    "Tonga": "Oceania",
    "Vanuatu": "Oceania",
    "United States Minor Outlying Islands": "Oceania",
    "French Polynesia": "Oceania",
    "Solomon Islands": "Oceania",
    "Cook Islands": "Oceania",
    "Marshall Islands": "Oceania",
    "Tasmania": "Oceania",

    # Antarctica
    "Antarctica": "Antarctica",

    # Water
    "Atlantic Ocean": "Water",
    "Pacific Ocean": "Water",
    "North Sea": "Water",
    "Baltic Sea": "Water",
    "Black Sea": "Water",
    "Mediterranean Sea": "Water",
    "Philippine Sea": "Water",
    "Andaman Sea": "Water",
    "Persian Gulf": "Water",
    "Gulf of Sirte": "Water",
    "Gulf of Tonkin": "Water",
    "English Channel": "Water",
    "Taiwan Strait": "Water",

    # Islands / Territories
    "Newfoundland": "Island",
    "Canary Islands": "Island",
    "Azores": "Island",
    "West Indies": "Island",
    "Leeward Islands": "Island",
    "Crete": "Island",
    "Okinawa": "Island",
    "Eugene Island": "Island",
    "Islay Island": "Island",
    "Mariana Islands": "Island",
    "Borneo": "Island",
    "Grenadines": "Island",
    "Great Inagua": "Island",
    "Bimini": "Island",
    "Caribbean": "Island"
}

df["Region"] = df["Country"].map(region_lookup)

# anything unmapped becomes Unknown
df["Region"] = df["Region"].fillna("Unknown")

print(df["Region"].value_counts())

Region
North America    1625
Europe            999
Asia              869
South America     560
Africa            432
Oceania           134
Water              66
Island             34
Unknown            21
Antarctica          1
Name: count, dtype: int64


In [340]:
# final check
display(
    df.loc[df["Region"] == "Unknown", "Country"]
      .value_counts()
      .head(100)
)

Series([], Name: count, dtype: int64)

## Report of section 1. Pre-processing

Date: No nulls, datatype changed from object -> datetime64

Time: 42% of values are null. 15 values had a string length not equal to 5, and upon inspection, they all looked like genuine recording errors. Mainly, the had 'c' or 'c:' at the front, which could denote circa or clock. Some of them were missing the first '0', for example '1:00' instead of '01:00'. All values were replaced with their time value in a valid format, except one value '114:20'. It appears that a number has been typed in by mistake at the start, however, it is impossible to judge which number has been mistyped, so this was replaced with null and subsequently 'Unknown' for Time_period.

Location: